# LettuceDetect baseline on `Ivan1008/toolace-hallucination-spans`

Implements task **(2) Evaluate existing baseline methods** from `Hallucination Detection in Tool Calling.pdf` using the encoder-based LettuceDetect model from Kovács & Recski, 2025 ([arXiv:2502.17125](https://arxiv.org/abs/2502.17125)).

**Approach (matches the LettuceDetect paper):**
- Token-classification model `KRLabsOrg/lettucedect-{base,large}-modernbert-en-v1`, ModernBERT backbone, 4096-token context window.
- Inputs are formatted as `(context, question, answer)` triples — here `context = tool output`, `question = user query`, `answer = model final response`, exactly as the task description requires.
- Tokens belonging to the answer are classified as `0=supported` / `1=hallucinated`. Spans are built by collapsing consecutive `1`-tokens.
- We evaluate **two granularities** (Tables 2 and 3 of the Lettuce paper):
  - **Example-level** — binary detection: does this record contain *any* hallucination?
  - **Span-level** — character-overlap precision/recall/F1 between predicted spans and gold spans within the answer (RAGTruth methodology).

**Dataset:** `Ivan1008/toolace-hallucination-spans` on the Hugging Face Hub. We evaluate the four published configs (`combined`, `contradiction`, `missing_tool`, `overgeneration`) on the `test` split.

**Baselines reported in the resulting table:**
1. `lettucedect-base-modernbert-en-v1` (≈150M params)
2. `lettucedect-large-modernbert-en-v1` (≈396M params)
3. A *lexical* sanity baseline (a token in the answer is hallucinated iff it does not appear in the context) — this is *not* from the Lettuce paper, but is included as the trivial floor every encoder must beat.

Both Lettuce checkpoints were trained on RAGTruth (news/QA/data-to-text), so this is a strict **zero-shot domain-transfer** evaluation onto tool-calling dialogues.

## 1. Setup

In [1]:
!pip install lettucedetect

In [2]:
import subprocess, sys as _sys
# subprocess.check_call([
#     _sys.executable, "-m", "pip", "install", "-q",
#     "datasets>=3.2", "lettucedetect>=0.1.7",
#     "transformers>=4.48", "huggingface_hub>=0.24",
#     "pyarrow>=15", "pandas>=2.0", "tqdm",
# ])

import json
import os
import re
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from tqdm.auto import tqdm

from lettucedetect.models.inference import HallucinationDetector

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"torch: {torch.__version__}  datasets installed  device: {DEVICE}")

DATASET_REPO = "Ivan1008/toolace-hallucination-spans"
CONFIGS = ["combined", "contradiction", "missing_tool", "overgeneration"]
SPLIT = "test"
MODEL_IDS = [
    "KRLabsOrg/lettucedect-base-modernbert-en-v1",
    "KRLabsOrg/lettucedect-large-modernbert-en-v1",
]
MAX_RECORDS_PER_CONFIG = None  # set e.g. 50 for a quick smoke test
OUTPUT_DIR = Path("results/lettucedetect_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch: 2.10.0+cu128  datasets installed  device: cuda


## 2. Load the dataset

Each record is RAGTruth-shaped:
- `query` — user query → used as `question` for LettuceDetect
- `context` — serialized tool output(s) → used as `context`
- `output` — assistant final answer → used as `answer` (predictions are over its character offsets)
- `hallucination_labels` — list of `{start, end, text, label}` over `output[start:end]`
- `meta.corruption_type` — one of `clean`, `contradiction`, `missing_tool`, `overgeneration`

In [3]:
def list_split_parquets(repo_id: str, config: str, split: str) -> list[str]:
    """Return hf:// URIs for all parquet files of a given config/split."""
    files = HfApi().list_repo_files(repo_id, repo_type="dataset")
    needles = (f"{config}/{split}-", f"{config}/{split}.parquet")
    return [
        f"hf://datasets/{repo_id}/{f}"
        for f in files
        if f.endswith(".parquet") and any(n in f for n in needles)
    ]


def load_config(repo_id: str, config: str, split: str) -> list[dict]:
    """Load (config, split) into a plain list of dicts.

    Tries the canonical ``load_dataset`` path first; falls back to direct
    parquet loading if the installed ``datasets`` cannot parse the dataset's
    feature schema (e.g. the ``Json`` feature type on older ``datasets``).
    """
    try:
        ds = load_dataset(repo_id, config, split=split)
        return [dict(row) for row in ds]
    except (ValueError, TypeError) as e:
        print(f"  load_dataset failed for {config}: {e}; falling back to parquet")
        uris = list_split_parquets(repo_id, config, split)
        if not uris:
            raise FileNotFoundError(f"no parquet files found for {repo_id}:{config}/{split}") from e
        df = pd.concat([pd.read_parquet(u) for u in uris], ignore_index=True)
        records = df.to_dict(orient="records")
        for r in records:
            for k in ("hallucination_labels", "meta"):
                if isinstance(r.get(k), str):
                    r[k] = json.loads(r[k])
                elif hasattr(r.get(k), "tolist"):
                    r[k] = r[k].tolist()
        return records


datasets_by_config: dict[str, list[dict]] = {}
for cfg in CONFIGS:
    print(f"loading {cfg}/{SPLIT} ...")
    records = load_config(DATASET_REPO, cfg, SPLIT)
    if MAX_RECORDS_PER_CONFIG:
        records = records[:MAX_RECORDS_PER_CONFIG]
    datasets_by_config[cfg] = records

rows = []
for cfg, records in datasets_by_config.items():
    types = Counter(r["meta"]["corruption_type"] for r in records)
    rows.append({
        "config": cfg,
        "n": len(records),
        "clean": types.get("clean", 0),
        "contradiction": types.get("contradiction", 0),
        "missing_tool": types.get("missing_tool", 0),
        "overgeneration": types.get("overgeneration", 0),
    })
pd.DataFrame(rows).set_index("config")

loading combined/test ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

combined/train-00000-of-00001.parquet:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

combined/validation-00000-of-00001.parqu(…):   0%|          | 0.00/230k [00:00<?, ?B/s]

...

overgeneration/train-00000-of-00001.parq(…):   0%|          | 0.00/1.47M [00:00<?, ?B/s]

overgeneration/validation-00000-of-00001(…):   0%|          | 0.00/193k [00:00<?, ?B/s]

overgeneration/test-00000-of-00001.parqu(…):   0%|          | 0.00/193k [00:00<?, ?B/s]

  load_dataset failed for overgeneration: Feature type 'Json' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'List', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf']; falling back to parquet


,n,clean,contradiction,missing_tool,overgeneration
config,,,,,
combined,235,31,36,70,98
contradiction,67,31,36,0,0
missing_tool,101,31,0,70,0
overgeneration,129,31,0,0,98


## 3. Metrics

**Example-level** (Lettuce Table 2): a record is *positive* iff it has any hallucination. Predicted positive iff the model emits at least one span. Compute precision/recall/F1 over records.

**Span-level** (Lettuce Table 3): RAGTruth-style character-overlap. For each record, the *gold* mask is `True` for character indices covered by any `hallucination_labels` span; the *pred* mask is `True` for character indices covered by any predicted span. Then aggregate TP/FP/FN over characters across the whole split (micro-average).

Both are exactly the two reporting protocols used in the LettuceDetect paper.

In [4]:
def char_mask(text: str, spans: list[dict]) -> list[bool]:
    mask = [False] * len(text)
    for s in spans:
        start = max(0, int(s.get("start", 0)))
        end = min(len(text), int(s.get("end", 0)))
        for i in range(start, end):
            mask[i] = True
    return mask


def prf1(tp: int, fp: int, fn: int) -> dict:
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"precision": p, "recall": r, "f1": f, "tp": tp, "fp": fp, "fn": fn}


def example_level(per_record: list[tuple[bool, bool]]) -> dict:
    """per_record: list of (gold_positive, pred_positive)."""
    tp = sum(1 for g, p in per_record if g and p)
    fp = sum(1 for g, p in per_record if (not g) and p)
    fn = sum(1 for g, p in per_record if g and (not p))
    out = prf1(tp, fp, fn)
    out["n"] = len(per_record)
    out["n_positive"] = sum(1 for g, _ in per_record if g)
    return out


def span_level(per_record_masks: list[tuple[list[bool], list[bool]]]) -> dict:
    """per_record_masks: list of (gold_mask, pred_mask) over answer characters."""
    tp = fp = fn = 0
    for gold, pred in per_record_masks:
        for g, p in zip(gold, pred):
            if p and g:
                tp += 1
            elif p and not g:
                fp += 1
            elif g and not p:
                fn += 1
    out = prf1(tp, fp, fn)
    out["n"] = len(per_record_masks)
    return out

## 4. Lexical floor baseline

A trivial baseline: predict every content word in the answer that does not appear in the context as hallucinated. This is a span-level baseline only; it is the *minimum* an actual model must beat.

In [5]:
WORD_RE = re.compile(r"[A-Za-z0-9]+")


def words_set(text: str) -> set[str]:
    return {m.group(0).lower() for m in WORD_RE.finditer(text) if len(m.group(0)) > 2}


def lexical_predict_mask(context: str, output: str) -> list[bool]:
    ctx = words_set(context)
    mask = [False] * len(output)
    for m in WORD_RE.finditer(output):
        tok = m.group(0).lower()
        if len(tok) <= 2:
            continue
        if tok not in ctx:
            for i in range(m.start(), m.end()):
                mask[i] = True
    return mask


def lexical_predict_spans(context: str, output: str) -> list[dict]:
    """Convert the lexical mask into spans for example-level scoring."""
    mask = lexical_predict_mask(context, output)
    spans = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            spans.append({"start": i, "end": j, "text": output[i:j]})
            i = j
        else:
            i += 1
    return spans

## 5. LettuceDetect inference

Wrap `HallucinationDetector(method="transformer", model_path=...)` and run it on every record. We collect:
- predicted spans `{start, end, text, confidence}` over the answer's character offsets;
- gold spans from `hallucination_labels`;
- per-record positive/negative flags for example-level scoring.

Long records that exceed 4096 tokens are skipped by LettuceDetect's tokenizer (it raises). We catch and treat as *no prediction* (predicted negative).

In [6]:
def run_lettuce(records: list[dict], detector: HallucinationDetector) -> list[dict]:
    """Run inference and return list of dicts with gold/pred spans + masks."""
    results = []
    n_failed = 0
    for r in tqdm(records, leave=False):
        out = r["output"]
        try:
            spans = detector.predict(
                context=[r["context"]],
                question=r["query"],
                answer=out,
                output_format="spans",
            )
        except Exception:
            spans = []
            n_failed += 1
        results.append({
            "id": r.get("id"),
            "corruption_type": r["meta"]["corruption_type"],
            "output": out,
            "gold_spans": r["hallucination_labels"],
            "pred_spans": spans,
        })
    if n_failed:
        print(f"  {n_failed} records failed inference (treated as no-prediction)")
    return results

## 6. Score a result set

Given a list of per-record `{gold_spans, pred_spans, output, corruption_type}`, produce overall + per-corruption-type example-level and span-level metrics.

In [7]:
def score_results(per_record: list[dict]) -> dict:
    by_type_ex: dict[str, list[tuple[bool, bool]]] = defaultdict(list)
    by_type_sp: dict[str, list[tuple[list[bool], list[bool]]]] = defaultdict(list)
    overall_ex: list[tuple[bool, bool]] = []
    overall_sp: list[tuple[list[bool], list[bool]]] = []

    for r in per_record:
        gold_pos = bool(r["gold_spans"])
        pred_pos = bool(r["pred_spans"])
        gold_mask = char_mask(r["output"], r["gold_spans"])
        pred_mask = char_mask(r["output"], r["pred_spans"])
        overall_ex.append((gold_pos, pred_pos))
        overall_sp.append((gold_mask, pred_mask))
        by_type_ex[r["corruption_type"]].append((gold_pos, pred_pos))
        by_type_sp[r["corruption_type"]].append((gold_mask, pred_mask))

    return {
        "example_level": {
            "overall": example_level(overall_ex),
            "by_type": {t: example_level(v) for t, v in by_type_ex.items()},
        },
        "span_level": {
            "overall": span_level(overall_sp),
            "by_type": {t: span_level(v) for t, v in by_type_sp.items()},
        },
    }

## 7. Run all baselines on all configs

For each model and each config, run inference and score. Persist raw predictions to disk so the table can be regenerated without re-running the model.

In [8]:
all_metrics: dict[str, dict[str, dict]] = {}

print("==== Lexical floor baseline ====")
lex_metrics: dict[str, dict] = {}
for cfg, records in datasets_by_config.items():
    per_record = []
    for r in records:
        spans = lexical_predict_spans(r["context"], r["output"])
        per_record.append({
            "id": r.get("id"),
            "corruption_type": r["meta"]["corruption_type"],
            "output": r["output"],
            "gold_spans": r["hallucination_labels"],
            "pred_spans": spans,
        })
    lex_metrics[cfg] = score_results(per_record)
    ex = lex_metrics[cfg]["example_level"]["overall"]
    sp = lex_metrics[cfg]["span_level"]["overall"]
    print(f"  {cfg:14s}  example F1={ex['f1']:.3f}  span F1={sp['f1']:.3f}  (n={ex['n']})")

all_metrics["lexical"] = lex_metrics
(OUTPUT_DIR / "metrics_lexical.json").write_text(json.dumps(lex_metrics, indent=2))

==== Lexical floor baseline ====
  combined        example F1=0.932  span F1=0.238  (n=235)
  contradiction   example F1=0.706  span F1=0.048  (n=67)
  missing_tool    example F1=0.824  span F1=0.140  (n=101)
  overgeneration  example F1=0.867  span F1=0.295  (n=129)


6567

In [9]:
for model_id in MODEL_IDS:
    print(f"\n==== {model_id} ====")
    t0 = time.time()
    detector = HallucinationDetector(method="transformer", model_path=model_id)
    print(f"  loaded in {time.time() - t0:.1f}s")

    model_metrics: dict[str, dict] = {}
    short = model_id.split("/")[-1]
    raw_dir = OUTPUT_DIR / "raw" / short
    raw_dir.mkdir(parents=True, exist_ok=True)

    for cfg, records in datasets_by_config.items():
        print(f"  config={cfg}  n={len(records)}")
        t1 = time.time()
        per_record = run_lettuce(records, detector)
        elapsed = time.time() - t1
        rate = len(records) / elapsed if elapsed else 0.0
        with (raw_dir / f"{cfg}_{SPLIT}.jsonl").open("w") as f:
            for r in per_record:
                f.write(json.dumps(r) + "\n")
        m = score_results(per_record)
        model_metrics[cfg] = m
        ex = m["example_level"]["overall"]
        sp = m["span_level"]["overall"]
        print(
            f"    example  P={ex['precision']:.3f} R={ex['recall']:.3f} F1={ex['f1']:.3f}  |  "
            f"span  P={sp['precision']:.3f} R={sp['recall']:.3f} F1={sp['f1']:.3f}  |  "
            f"{rate:.1f} rec/s"
        )

    all_metrics[short] = model_metrics
    (OUTPUT_DIR / f"metrics_{short}.json").write_text(json.dumps(model_metrics, indent=2))

    del detector
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

(OUTPUT_DIR / "metrics_all.json").write_text(json.dumps(all_metrics, indent=2))


==== KRLabsOrg/lettucedect-base-modernbert-en-v1 ====


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

  0%|          | 0/101 [00:00<?, ?it/s]

    example  P=0.812 R=0.986 F1=0.890  |  span  P=0.118 R=0.910 F1=0.209  |  7.7 rec/s
  config=overgeneration  n=129


  0%|          | 0/129 [00:00<?, ?it/s]

    example  P=0.855 R=0.959 F1=0.904  |  span  P=0.210 R=0.690 F1=0.322  |  7.3 rec/s


22144

## 8. Results tables

Reproduce the **Lettuce-paper-style** layout: rows = method, columns = `{config} × {Precision, Recall, F1}` + an `overall` block.

We emit two tables, one for each granularity (mirroring Tables 2 and 3 of the paper).

In [10]:
def build_table(all_metrics: dict, granularity: str) -> pd.DataFrame:
    """granularity in {'example_level', 'span_level'}"""
    rows = []
    for method, by_cfg in all_metrics.items():
        row = {"method": method}
        for cfg in CONFIGS:
            m = by_cfg[cfg][granularity]["overall"]
            row[(cfg, "P")] = round(m["precision"], 4)
            row[(cfg, "R")] = round(m["recall"], 4)
            row[(cfg, "F1")] = round(m["f1"], 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("method")
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    return df


example_table = build_table(all_metrics, "example_level")
span_table = build_table(all_metrics, "span_level")

print("Example-level (Table 2 analogue):")
display(example_table)
print("\nSpan-level char-overlap (Table 3 analogue):")
display(span_table)

example_table.to_csv(OUTPUT_DIR / "example_level.csv")
span_table.to_csv(OUTPUT_DIR / "span_level.csv")

Example-level (Table 2 analogue):


combined                 contradiction  \
                                          P       R      F1             P   
method                                                                      
lexical                              0.8718  1.0000  0.9315        0.5455   
lettucedect-base-modernbert-en-v1    0.9220  0.9265  0.9242        0.6596   
lettucedect-large-modernbert-en-v1   0.9234  0.9461  0.9346        0.6522   

                                                   missing_tool          \
                                         R      F1            P       R   
method                                                                    
lexical                             1.0000  0.7059       0.7000  1.0000   
lettucedect-base-modernbert-en-v1   0.8611  0.7470       0.7975  0.9000   
lettucedect-large-modernbert-en-v1  0.8333  0.7317       0.8118  0.9857   

                                           overgeneration                  
                                        F1              P       R      F1  
method                                                                     
lexical                             0.8235         0.7656  1.0000  0.8673  
lettucedect-base-modernbert-en-v1   0.8456         0.8559  0.9694  0.9091  
lettucedect-large-modernbert-en-v1  0.8903         0.8545  0.9592  0.9038


Span-level char-overlap (Table 3 analogue):


combined                 contradiction  \
                                          P       R      F1             P   
method                                                                      
lexical                              0.1437  0.6975  0.2383        0.0249   
lettucedect-base-modernbert-en-v1    0.1761  0.7051  0.2818        0.0357   
lettucedect-large-modernbert-en-v1   0.1772  0.7377  0.2857        0.0425   

                                                   missing_tool          \
                                         R      F1            P       R   
method                                                                    
lexical                             0.7739  0.0483       0.0778  0.7086   
lettucedect-base-modernbert-en-v1   0.6382  0.0677       0.1095  0.7937   
lettucedect-large-modernbert-en-v1  0.7060  0.0801       0.1179  0.9102   

                                           overgeneration                  
                                        F1              P       R      F1  
method                                                                     
lexical                             0.1401         0.1875  0.6919  0.2950  
lettucedect-base-modernbert-en-v1   0.1925         0.2154  0.6824  0.3275  
lettucedect-large-modernbert-en-v1  0.2087         0.2104  0.6903  0.3224

## 9. Per-corruption-type breakdown (combined config)

The `combined` config contains `clean`, `contradiction`, `missing_tool`, and `overgeneration` records. Breaking the metrics down by `meta.corruption_type` shows which hallucination kinds each baseline catches and which it misses.

For *clean* records, span-level recall/F1 is undefined (no gold positives), so only example-level numbers are meaningful (precision ≡ 1 - false-positive-rate).


In [11]:
def build_by_type_table(all_metrics: dict, granularity: str, cfg: str = "combined") -> pd.DataFrame:
    rows = []
    for method, by_cfg in all_metrics.items():
        by_type = by_cfg[cfg][granularity]["by_type"]
        for ctype, m in by_type.items():
            rows.append({
                "method": method,
                "corruption_type": ctype,
                "n": m["n"],
                "precision": round(m["precision"], 4),
                "recall": round(m["recall"], 4),
                "f1": round(m["f1"], 4),
            })
    return pd.DataFrame(rows).pivot_table(
        index="method",
        columns="corruption_type",
        values=["precision", "recall", "f1"],
        aggfunc="first",
    )


print("Per-type, example-level (combined config):")
display(build_by_type_table(all_metrics, "example_level", "combined"))
print("\nPer-type, span-level (combined config):")
display(build_by_type_table(all_metrics, "span_level", "combined"))

Per-type, example-level (combined config):


f1                             \
corruption_type                    clean contradiction missing_tool   
method                                                                
lettucedect-base-modernbert-en-v1    0.0        0.9254       0.9474   
lettucedect-large-modernbert-en-v1   0.0        0.9091       0.9928   
lexical                              0.0        1.0000       1.0000   

                                                  precision                \
corruption_type                    overgeneration     clean contradiction   
method                                                                      
lettucedect-base-modernbert-en-v1          0.9845       0.0           1.0   
lettucedect-large-modernbert-en-v1         0.9792       0.0           1.0   
lexical                                    1.0000       0.0           1.0   

                                                               recall  \
corruption_type                    missing_tool overgeneration  clean   
method                                                                  
lettucedect-base-modernbert-en-v1           1.0            1.0    0.0   
lettucedect-large-modernbert-en-v1          1.0            1.0    0.0   
lexical                                     1.0            1.0    0.0   

                                                                              
corruption_type                    contradiction missing_tool overgeneration  
method                                                                        
lettucedect-base-modernbert-en-v1         0.8611       0.9000         0.9694  
lettucedect-large-modernbert-en-v1        0.8333       0.9857         0.9592  
lexical                                   1.0000       1.0000         1.0000


Per-type, span-level (combined config):


f1                             \
corruption_type                    clean contradiction missing_tool   
method                                                                
lettucedect-base-modernbert-en-v1    0.0        0.1179       0.2169   
lettucedect-large-modernbert-en-v1   0.0        0.1470       0.2335   
lexical                              0.0        0.0838       0.1658   

                                                  precision                \
corruption_type                    overgeneration     clean contradiction   
method                                                                      
lettucedect-base-modernbert-en-v1          0.3493       0.0        0.0650   
lettucedect-large-modernbert-en-v1         0.3432       0.0        0.0820   
lexical                                    0.3254       0.0        0.0443   

                                                               recall  \
corruption_type                    missing_tool overgeneration  clean   
method                                                                  
lettucedect-base-modernbert-en-v1        0.1256         0.2347    0.0   
lettucedect-large-modernbert-en-v1       0.1339         0.2284    0.0   
lexical                                  0.0939         0.2128    0.0   

                                                                              
corruption_type                    contradiction missing_tool overgeneration  
method                                                                        
lettucedect-base-modernbert-en-v1         0.6382       0.7937         0.6824  
lettucedect-large-modernbert-en-v1        0.7060       0.9102         0.6903  
lexical                                   0.7739       0.7086         0.6919

## 10. Qualitative inspection

Show a handful of records from `combined/test` with their gold spans and the predictions of the *large* LettuceDetect model. Useful for spotting failure modes (missed `missing_tool` insertions, partial `contradiction` recall, false positives on JSON literals).

In [12]:
import html
from IPython.display import HTML


def highlight(output: str, gold: list[dict], pred: list[dict]) -> str:
    g = char_mask(output, gold)
    p = char_mask(output, pred)
    parts = []
    for i, ch in enumerate(output):
        gg, pp = g[i], p[i]
        ch = html.escape(ch)
        if gg and pp:
            parts.append(f"<span style='background:#9f9'>{ch}</span>")  # TP
        elif gg:
            parts.append(f"<span style='background:#fdd;text-decoration:underline'>{ch}</span>")  # FN
        elif pp:
            parts.append(f"<span style='background:#ffd'>{ch}</span>")  # FP
        else:
            parts.append(ch)
    return "".join(parts)


large_short = MODEL_IDS[-1].split("/")[-1]
raw_path = OUTPUT_DIR / "raw" / large_short / f"combined_{SPLIT}.jsonl"
if raw_path.exists():
    sample_records = [json.loads(l) for l in raw_path.read_text().splitlines()]
    examples = [r for r in sample_records if r["corruption_type"] != "clean"][:5]
    blocks = [
        "<style>body{font-family:system-ui}</style>"
        "<p><b>Legend:</b> "
        "<span style='background:#9f9'>TP</span> "
        "<span style='background:#fdd;text-decoration:underline'>FN (gold only)</span> "
        "<span style='background:#ffd'>FP (pred only)</span></p>"
    ]
    for r in examples:
        blocks.append(f"<hr><b>{r['id']}</b> — corruption_type=<code>{r['corruption_type']}</code>")
        blocks.append(f"<div style='white-space:pre-wrap'>{highlight(r['output'], r['gold_spans'], r['pred_spans'])}</div>")
    display(HTML("\n".join(blocks)))
else:
    print(f"raw predictions not found at {raw_path}; rerun the inference cell first.")

## 11. Discussion

What this notebook delivers, against the task spec from `Hallucination Detection in Tool Calling.pdf` (§2: *Evaluate existing baseline methods*):

- **Method.** LettuceDetect (§4 of Kovács & Recski, 2025) — ModernBERT-based token classifier on a `(context, question, answer)` triple, span-collapsed at threshold 0.5.
- **Mapping.** `query → question`, `tool output → context`, `assistant final response → answer`, exactly as specified in §2 of the task PDF.
- **Datasets.** The four published configs of `Ivan1008/toolace-hallucination-spans` on the `test` split (235 / 67 / 101 / 129 records for `combined` / `contradiction` / `missing_tool` / `overgeneration`).
- **Reporting.** Two tables in the Lettuce-paper layout: example-level binary detection and span-level character-overlap micro-F1.
- **Floor.** Lexical "word-not-in-context" baseline.

### Measured results

**Example-level F1 (overall, by config):**

| Method | combined | contradiction | missing_tool | overgeneration |
|---|---:|---:|---:|---:|
| lexical                            | 0.932 | 0.706 | 0.824 | 0.867 |
| lettucedect-base-modernbert-en-v1  | 0.924 | 0.747 | 0.846 | 0.909 |
| lettucedect-large-modernbert-en-v1 | **0.935** | 0.732 | **0.890** | 0.904 |

**Span-level character-overlap F1 (overall, by config):**

| Method | combined | contradiction | missing_tool | overgeneration |
|---|---:|---:|---:|---:|
| lexical                            | 0.238 | 0.048 | 0.140 | 0.295 |
| lettucedect-base-modernbert-en-v1  | 0.282 | 0.068 | 0.193 | **0.328** |
| lettucedect-large-modernbert-en-v1 | **0.286** | **0.080** | **0.209** | 0.322 |

### What these numbers say

1. **Example-level detection is largely solved already by the lexical floor.** `combined` has 31 clean records out of 235 (~13 %), so the prior is heavily skewed toward "positive". The encoder still beats the floor on `combined` (0.935 vs 0.932 F1) and `overgeneration` (0.904 vs 0.867), but the gap is small — *finding* a hallucination in this dataset is easy; *localizing* it is the hard problem.

2. **Span-level F1 is uniformly low across the board** (best 0.286 on `combined`). This is consistent with the dataset card's pre-audit validation numbers (0.198 for the `combined` validation split) and confirms that out-of-domain transfer from RAGTruth (news / QA / data-to-text prose) to ToolACE (JSON-serialized tool output) is the main blocker. The encoder still consistently beats the lexical floor at span level — by **+5 pp** on `combined`, **+7 pp** on `missing_tool`, **+3 pp** on the others — so it does carry signal beyond surface lexical mismatch.

3. **High span recall, low span precision** (large model: `combined` P=0.18 / R=0.74; `missing_tool` P=0.12 / R=**0.91**). LettuceDetect is sensitive to the *general region* of the hallucination but over-predicts: it flags whole sentences instead of the swapped token. This is the dominant failure mode in the qualitative panel (yellow FP zones around the green TP cores).

4. **`contradiction` is the hardest type by a wide margin** (span F1 0.080 large vs 0.295 on `overgeneration`). Replaced tokens (e.g. one city swapped for another) are individually unremarkable to a token classifier trained on prose — the model marks the surrounding sentence as suspicious but doesn't isolate the swapped value. Span recall is 0.71 for the large model, but precision collapses to 0.04.

5. **`missing_tool` benefits most from the larger model** (span F1 0.193 → 0.209, span recall 0.79 → **0.91**). Detecting an "off-policy" assistant offer is closer in flavour to detecting a non-grounded sentence in RAGTruth, which is the regime ModernBERT-large was actually trained on.

6. **`overgeneration` is the easiest** (span F1 0.328 base, 0.322 large). Inserted sentences contain content words absent from the short, structured tool response, and even the lexical baseline gets 0.295 here.

7. **Base vs large is a small delta on this domain** (overall span F1 +0.4 pp on `combined`, ~+1–2 pp per config). The bottleneck is *training-data domain*, not parameter count — both checkpoints were trained on the same RAGTruth corpus.

### Implications for §3 (Improve the baseline approaches)

These numbers define the floor that step §3 of the task must beat. Concrete leads suggested by the breakdowns above:

- **Fine-tune on the train split of this dataset.** The largest gap is span-level precision (~0.04–0.21). The encoder already finds the right region; supervised fine-tuning on tool-calling JSON contexts should be the single biggest lever. The Lettuce repo's trainer expects RAGTruth-format data, which is exactly the schema we publish here — almost a drop-in.
- **Inject the available-tools list into the context** (currently only the *responses* are passed). This is specifically what `missing_tool` requires: the encoder cannot know an action is off-policy without seeing the tool catalog. Expected lift on `missing_tool` precision in particular.
- **Calibrate the per-token threshold.** The encoder's recall is 0.7–0.9 but precision is 0.04–0.21. A per-config threshold sweep (or learned calibration on the validation split) could trade off recall for precision and likely lift span F1 by several points without retraining.
- **Add a contradiction-aware contrast objective** (or an entity-level NLI head). Plain token classification doesn't have an inductive bias for "this *specific* value disagrees with the JSON" — a span-vs-context entailment objective would.

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
